# 9장 (실습 4): 패치를 자동으로 배포하고, 결과를 확인하고, 패치 전에는 Workaround로 막기

> 책 『AI 보안, 클라우드에서 시작한다』 9장 [실습] 4번을 노트북에서 단계별로 실행하는 버전입니다. 서버 3대로 이루어진 작은 서비스에 보안 공지가 들어온 상황에서, 9-4의 세 가지 질문을 직접 해 봅니다. 소요 시간은 30~40분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 패치 전 **Workaround를 적용하고 그 효과를 확인**하며, Workaround로는 막지 못하는 경로가 있다는 것을 확인합니다.
2. 패치를 **Canary부터** 배포하고, 문제가 생기면 **자동으로 되돌리는** 흐름을 확인합니다.
3. "패치 완료" 보고가 아니라 **다시 수집하고 다시 점검한 결과**로 조치를 확인하고, 그 뒤에만 Workaround를 해제합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | 예정 |
| 1 | 환경 만들기와 현황 보기 | 예정 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | 예정 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | 예정 |
| 4 | "패치 완료" 보고를 믿지 않는다 | 예정 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | 예정 |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

## 실습 시나리오

주문 조회 서비스가 서버 3대(app1, app2, app3)에서 돌고 있고, 외부 요청은 앞단 프록시를 거쳐 세 서버로 나뉩니다. 어느 날 서비스 공급자가 보안 공지를 냈습니다. 버전 1.0의 관리 기능이 인증 없이 열려 있어 고객 정보를 내려받을 수 있고, 실제 악용도 확인되었다는 내용입니다. 수정 버전은 1.1입니다.

이 취약점(LAB-2026-0001)은 실습을 위해 만든 **가상 취약점**입니다. 실제 CVE와 스캐너로 하는 부분(취약점 정보 수집, 버전 대조)은 실습 1·3과 8장 실습 1에서 다뤘으므로, 이 실습은 그다음 단계인 **배포, 확인, 임시 조치의 관리**에 집중합니다.

```text
 외부 요청 ─▶ proxy(127.0.0.1:8080, Workaround 규칙 자리) ─┬─▶ app1 :8081  ← Canary
                                                           ├─▶ app2 :8082
                                                           └─▶ app3 :8083
 patch_cycle.py ─ status / workaround on·off / patch <버전> / report ─▶ state/ (티켓, Workaround, history.jsonl)
```

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker와 Python 3가 필요합니다. API Key는 필요 없습니다. 8080~8083 포트가 비어 있어야 합니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> - 2장 노트북([ch02_lab1_container_isolation.ipynb](../ch02-container-security/ch02_lab1_container_isolation.ipynb))의 실습 전 확인사항과 같습니다. `~/.venv-lab`에 jupyterlab을 설치한 뒤 이 폴더에서 `~/.venv-lab/bin/jupyter lab`을 실행합니다.
> - 노트북은 **반드시 `ch09-ops-tracing-sw-inventory` 폴더에서 열어야** 합니다. 실습 파일을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Python 3.12에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다.

## 0. 실습 준비와 실습 파일

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 환경 만들기와 현황 보기 | 예정 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | 예정 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | 예정 |
| 4 | "패치 완료" 보고를 믿지 않는다 | 예정 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | 예정 |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

docker     ✅ Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2
python3    ✅ Python 3.12.3


`[표 실습9-4]` 실습 4의 구성

| 파일 | 역할 |
|---|---|
| `patch-cycle/app/` | 주문 조회 서비스. 버전 1.0(관리 기능이 인증 없이 열림), 1.1(수정 버전), 1.2(수정은 됐지만 시작 점검이 실패하는 고장 난 패치) 세 가지 이미지로 빌드 |
| `patch-cycle/proxy/nginx.conf` | 앞단 프록시. Workaround 규칙이 들어갈 자리(`workaround.conf`)를 미리 비워 둠 |
| `patch-cycle/advisory.json` | 보안 공지. 영향 버전, 수정 버전, 실제 악용 여부, 권고된 Workaround |
| `patch-cycle/setup.sh` | 이미지 빌드, 서버 3대(1.0)와 프록시 실행, 패치 티켓 생성 |
| `patch-cycle/patch_cycle.py` | 상태 보기, Workaround 적용·해제, 패치 배포(Canary → 나머지, 실패 시 자동 롤백), 재수집·재점검으로 조치 확인, 지표 보고 |

| 이 실습 | 실무에서 같은 역할 (예) |
|---|---|
| Canary 배포와 헬스체크 | Kubernetes 롤링 업데이트와 준비 상태 점검(readiness probe), 배포 도구의 Canary 단계 |
| 자동 롤백 | 이전 이미지·스냅샷으로 되돌리는 배포 도구 기능 |
| 재수집·재점검 | 9-1의 자산·SW 수집 도구, 취약점 스캐너의 재스캔 |
| 티켓 | ITSM의 패치 티켓 |
| Workaround 등록 | 예외·임시 조치 관리 대장, WAF·프록시 규칙의 코드 관리 |

보안 공지를 먼저 읽어 봅니다.

In [2]:
WORK = Path.home() / "patch-lab"
sh(f"rm -rf {WORK} && cp -r '{LAB / 'patch-cycle'}' {WORK}")
print((WORK / "advisory.json").read_text())


def pc(*args):
    """patch_cycle.py를 실행하고 출력을 보여 줍니다. 터미널의 python3 patch_cycle.py ...와 같습니다."""
    r = subprocess.run(["python3", "patch_cycle.py", *args], cwd=WORK, capture_output=True, text=True)
    print(r.stdout + r.stderr)

{
  "id": "LAB-2026-0001",
  "title": "주문 조회 서비스 관리 기능 인증 우회 (실습용 가상 취약점)",
  "affected": ["1.0"],
  "fixed": "1.1",
  "exploited": true,
  "workaround": {
    "description": "패치 전까지 앞단 프록시에서 /admin/ 경로를 차단",
    "nginx": "location /admin/ { return 403 \"blocked by workaround LAB-2026-0001\\n\"; }"
  },
  "check": "/admin/export 요청에 200이 오면 취약"
}



## 1. 환경 만들기와 현황 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | 예정 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | 예정 |
| 4 | "패치 완료" 보고를 믿지 않는다 | 예정 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | 예정 |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

`setup.sh`가 이미지 3개를 빌드하고, 서버 3대를 모두 취약한 1.0으로 띄우고, 프록시를 띄운 뒤 보안 공지를 받아 패치 티켓을 엽니다. 처음에는 이미지 빌드에 1분쯤 걸립니다. 몇 번을 다시 실행해도 같은 처음 상태로 돌아갑니다.

In [3]:
%%bash
docker pull -q nginx:1.27-alpine >/dev/null          # 프록시 이미지와 2.1에서 쓸 curl 이미지를 미리 받아 둔다
docker pull -q curlimages/curl:8.10.1 >/dev/null
cd ~/patch-lab && chmod +x setup.sh && ./setup.sh

보안 공지 수신: LAB-2026-0001 주문 조회 서비스 관리 기능 인증 우회 (실습용 

가상 취약점)
  영향 버전 1.0, 수정 버전 1.1, 실제 악용 확인
  → 패치 티켓 T-

0001 생성


In [4]:
pc("status")

서버   실행 중 버전  헬스  내부 점검
app1   1.0           정상  취약
app2   1.0           정상  취약
app3   1.0           정상  취약
외부 경로(프록시): 노출 — 공격 요청이 통함
티켓 T-0001: 열림



`status`는 세 서버에 직접 접속해 **실제로 돌고 있는 버전을 다시 수집**하고, 공격 요청(`/admin/export`)을 보내 **취약한지 다시 점검**합니다. 외부 경로는 프록시를 통해 같은 요청을 보내 확인합니다.

공격 요청이 통하면 무엇이 나가는지 직접 봅니다(가짜 고객 데이터입니다).

In [5]:
print(sh("curl -s 127.0.0.1:8080/admin/export"))

{"customers": [{"name": "고객A", "phone": "010-0000-0001"}], "note": "인증 없이 열린 관리 기능 (LAB-2026-0001)"}


## 2. 패치보다 먼저: Workaround 적용과 효과 확인

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 고장 난 패치는 자동으로 되돌린다 | 예정 |
| 4 | "패치 완료" 보고를 믿지 않는다 | 예정 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | 예정 |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

수정 버전이 있어도 시험과 배포에는 시간이 걸립니다. 그사이 공지에 실린 임시 조치(프록시에서 `/admin/` 경로 차단)부터 적용합니다.

In [6]:
pc("workaround", "on")

Workaround W-0001 적용: 패치 전까지 앞단 프록시에서 /admin/ 경로를 차단
  효과 확인: 외부에서 공격 요청 → 차단됨
  내부 점검: 서버 3대는 여전히 취약 (app1, app2, app3) — 취약점 자체는 그대로
  기한: 2026-10-30까지 (패치 확인 뒤 해제)



In [7]:
print("── 프록시에 들어간 Workaround 규칙")
print((WORK / "proxy" / "workaround.conf").read_text())

── 프록시에 들어간 Workaround 규칙
location /admin/ { return 403 "blocked by workaround LAB-2026-0001\n"; }



두 가지를 확인합니다.

- **적용했다고 막힌 것은 아닙니다.** 스크립트는 규칙을 넣은 뒤 외부에서 공격 요청을 다시 보내 실제로 막혔는지 확인하고, 그 결과를 Workaround 기록에 남깁니다.
- **Workaround는 길만 막을 뿐입니다.** 서버 3대는 여전히 취약합니다. 프록시를 거치지 않는 내부 경로나, 이미 안에 들어온 공격자에게는 그대로 열려 있습니다. 그래서 기한(이 실습은 7일)을 두고 패치를 서두릅니다.

### 2.1. 내부 경로에서는 어떨까

같은 내부 네트워크(`patchlab`)에 있는 다른 컨테이너, 즉 "이미 안에 들어온 공격자"의 자리에서 서버에 직접 공격 요청을 보냅니다.

In [8]:
%%bash
echo "── 외부(프록시 경유)"
curl -s -o /dev/null -w "HTTP %{http_code}\n" 127.0.0.1:8080/admin/export
echo "── 내부망의 다른 컨테이너에서 app2로 직접"
docker run --rm --network patchlab curlimages/curl:8.10.1 -s -w "\nHTTP %{http_code}\n" http://app2:8000/admin/export

── 외부(프록시 경유)


HTTP 403


── 내부망의 다른 컨테이너에서 app2로 직접


{"customers": [{"name": "고객A", "phone": "010-0000-0001"}], "note": "인증 없이 열린 관리 

기능 (LAB-2026-0001)"}


HTTP 200


외부에서는 403으로 막히지만, 내부망에서는 200과 함께 고객 데이터가 나옵니다. Workaround는 **패치까지 시간을 버는 것**이지 취약점을 없애는 것이 아닙니다.

## 3. 고장 난 패치는 자동으로 되돌린다

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | ✅ 완료 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | "패치 완료" 보고를 믿지 않는다 | 예정 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | 예정 |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

공급자가 먼저 1.2를 냈다고 가정합니다. 수정은 되었지만 설정 파일 형식이 바뀌어 우리 환경에서는 시작 점검(`/health`)이 실패하는 패치입니다.

In [9]:
pc("patch", "1.2")

패치 1.2 배포 (Canary: app1 → 나머지)
  app1: 1.2 배포 뒤 헬스체크 실패 → 자동 롤백
  app1: 1.0(으)로 되돌림, 헬스체크 통과
  → 나머지 서버에는 배포하지 않음
티켓 T-0001: 롤백됨, 담당자에게 원인 확인 요청



패치는 먼저 app1 한 대(Canary)에만 배포되고, 헬스체크가 실패하자 사람의 판단을 기다리지 않고 1.0으로 되돌아갔습니다. 나머지 두 대는 건드리지 않았으므로 서비스는 계속 돌고, 티켓은 "롤백됨: 원인 확인 필요"로 담당자에게 넘어갑니다. 9-4의 "④ 이상 탐지 시 자동 복구"와 "⑤ 사람의 검증과 재패치"입니다. 롤백된 건을 자동으로 다시 시도하지 않는 것도 중요합니다. 같은 실패를 되풀이할 뿐이기 때문입니다.

> 🙋 (생각해 보기)
> 1.2를 세 대에 한꺼번에 배포했다면 어떻게 되었을까요? 세 서버 모두 시작 점검에 실패하는 상태가 되고, 되돌리는 동안 서비스 전체가 영향을 받습니다. Canary는 "실패해도 한 대만"이 되게 하는 장치입니다.

## 4. "패치 완료" 보고를 믿지 않는다

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | ✅ 완료 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | ✅ 완료 |
| 4 | "패치 완료" 보고를 믿지 않는다 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | 예정 |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

원인을 확인한 담당자가 1.1을 배포합니다. 그런데 app3 담당자는 이미지만 준비하고 재시작을 빠뜨린 채 작업을 "완료"로 보고했습니다. `--forget app3`은 이 상황을 흉내 냅니다.

In [10]:
pc("patch", "1.1", "--forget", "app3")

패치 1.1 배포 (Canary: app1 → 나머지)
  app1: 1.0 → 1.1 배포, 헬스체크 통과
  app2: 1.0 → 1.1 배포, 헬스체크 통과
  app3: 작업 완료로 보고됨
조치 결과 확인 (재수집·재점검)
  app1: 실행 중 1.1, 내부 점검 안전 → 확인
  app2: 실행 중 1.1, 내부 점검 안전 → 확인
  app3: 실행 중 1.0, 내부 점검 취약 → 실패
티켓 T-0001: 닫지 않음 — 확인 실패 app3를 담당자에게 다시 배정



In [11]:
pc("status")

서버   실행 중 버전  헬스  내부 점검
app1   1.1           정상  안전
app2   1.1           정상  안전
app3   1.0           정상  취약
외부 경로(프록시): 차단 — 공격 요청이 막힘
티켓 T-0001: 조치 미완료: 확인 실패 app3
Workaround W-0001: 적용 중, 만료 2026-10-30 (7일 남음)



작업 기록만 보면 세 서버 모두 끝난 것처럼 보입니다. 그러나 배포 뒤 **실행 중인 버전을 다시 수집하고 공격 요청을 다시 보내 보니** app3는 여전히 1.0이고 취약합니다. 티켓은 닫히지 않고 "조치 미완료: 확인 실패 app3"로 남습니다. 외부 경로는 Workaround 덕분에 막혀 있어 겉으로는 문제가 보이지 않는다는 점도 눈여겨봅니다. 외부 스캐닝만으로 조치를 확인했다면 app3를 놓쳤을 것입니다.

이 상태에서 Workaround를 해제하려고 하면 거부됩니다.

In [12]:
pc("workaround", "off")

패치 조치가 확인되기 전에는 Workaround를 해제할 수 없습니다.



## 5. 다시 배포하고, 확인되면 Workaround 해제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | ✅ 완료 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | ✅ 완료 |
| 4 | "패치 완료" 보고를 믿지 않는다 | ✅ 완료 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 지표로 읽기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [13]:
pc("patch", "1.1")
pc("workaround", "off")
pc("status")

패치 1.1 배포 (Canary: app1 → 나머지)
  app1: 이미 1.1, 건너뜀
  app2: 이미 1.1, 건너뜀
  app3: 1.0 → 1.1 배포, 헬스체크 통과
조치 결과 확인 (재수집·재점검)
  app1: 실행 중 1.1, 내부 점검 안전 → 확인
  app2: 실행 중 1.1, 내부 점검 안전 → 확인
  app3: 실행 중 1.1, 내부 점검 안전 → 확인
티켓 T-0001: 종결 (모든 서버 확인 완료). 이제 Workaround를 해제할 수 있음



Workaround 해제
  해제 뒤 외부에서 공격 요청 → 차단됨 (패치로 막힘)



서버   실행 중 버전  헬스  내부 점검
app1   1.1           정상  안전
app2   1.1           정상  안전
app3   1.1           정상  안전
외부 경로(프록시): 차단 — 공격 요청이 막힘
티켓 T-0001: 종결: 확인 완료
Workaround W-0001: 해제됨 (05:34:31)



모든 서버가 확인을 통과해야 티켓이 닫히고, 그다음에야 Workaround를 해제합니다. 해제한 뒤에도 외부에서 공격 요청을 다시 보내, 이번에는 Workaround가 아니라 패치 때문에 막혔다는 것을 확인합니다. 2.1의 내부 경로도 다시 시험합니다.

In [14]:
print(sh("docker run --rm --network patchlab curlimages/curl:8.10.1 -s -w ' HTTP %{http_code}' http://app2:8000/admin/export"))

{"error": "관리자 인증 필요"} HTTP 403


## 6. 지표로 읽기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | ✅ 완료 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | ✅ 완료 |
| 4 | "패치 완료" 보고를 믿지 않는다 | ✅ 완료 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | ✅ 완료 |
| 6 | 지표로 읽기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [15]:
pc("report")

조치 지표 (이 실습은 초 단위로 진행했지만, 실무에서는 시간·일 단위로 잰다)
  공지 → Workaround 적용·효과 확인: 2초
  공지 → 조치 확인 완료(티켓 종결): 23초
  패치 시도 3회, 자동 롤백 1회
  확인 실패로 티켓을 닫지 못한 횟수: 1회 (완료 보고와 실제가 달랐던 서버: app3)
  해제되지 않은 Workaround: 0건, 그중 기한이 지난 것: 0건



### 6.1. 조치 타임라인

모든 일은 `state/history.jsonl`에 기록됩니다. 보고서의 숫자가 어디서 나왔는지 기록을 시간 순서로 펼쳐 봅니다.

In [16]:
from datetime import datetime

hist = [json.loads(l) for l in (WORK / "state" / "history.jsonl").read_text().splitlines()]
t0 = datetime.fromisoformat(hist[0]["time"])
rows = []
for e in hist:
    detail = {k: v for k, v in e.items() if k not in ("time", "event")}
    rows.append((f"+{(datetime.fromisoformat(e['time']) - t0).seconds}초", e["event"],
                 json.dumps(detail, ensure_ascii=False)[:90]))
table(["경과", "사건", "내용"], rows)

| 경과 | 사건 | 내용 |
|---|---|---|
| +0초 | advisory | {"id": "LAB-2026-0001"} |
| +2초 | workaround_on | {"id": "W-0001", "verified": true} |
| +2초 | ticket | {"state": "Workaround 적용: 외부 경로 차단, 패치 대기", "note": ""} |
| +4초 | patch_start | {"version": "1.2"} |
| +14초 | rollback | {"server": "app1", "version": "1.2", "back": "1.0"} |
| +14초 | ticket | {"state": "롤백됨: 원인 확인 필요", "note": "1.2 헬스체크 실패"} |
| +14초 | patch_start | {"version": "1.1"} |
| +17초 | deploy | {"server": "app1", "version": "1.1"} |
| +20초 | deploy | {"server": "app2", "version": "1.1"} |
| +20초 | deploy_reported | {"server": "app3", "version": "1.1"} |
| +20초 | verify | {"fails": ["app3"]} |
| +20초 | ticket | {"state": "조치 미완료: 확인 실패 app3", "note": ""} |
| +20초 | patch_start | {"version": "1.1"} |
| +23초 | deploy | {"server": "app3", "version": "1.1"} |
| +23초 | verify | {"fails": []} |
| +23초 | ticket | {"state": "종결: 확인 완료", "note": ""} |
| +24초 | workaround_off | {"exposed": false} |

`[표 실습9-5]` 실습 결과와 9장 지표의 연결

| 실습 결과 | 9장 지표 (표 9-15) | 실무에서 보는 방법 |
|---|---|---|
| 공지 → Workaround 적용·효과 확인 | Workaround 적용 시간 | 시간 단위 이내가 목표 |
| 공지 → 조치 확인 완료 | KEV 취약점 평균 조치 시간 | "작업 완료"가 아니라 "확인 완료" 시각으로 계산 |
| 패치 시도와 자동 롤백 | 롤백률과 재패치 시간 | 롤백률이 높으면 시험 환경과 운영 환경의 차이를 점검 |
| 확인 실패 | 자산 영역별 패치 준수율 | 확인 실패가 반복되는 영역(재시작이 필요한 서버, 어플라이언스)을 집중 관리 |
| 해제되지 않은 Workaround | 만료된 Workaround 수 | 0에 가깝게. 기한이 지나도 남아 있는 임시 규칙은 그 자체로 운영 위험 |

## 7. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | ✅ 완료 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | ✅ 완료 |
| 4 | "패치 완료" 보고를 믿지 않는다 | ✅ 완료 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | ✅ 완료 |
| 6 | 지표로 읽기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 정리 | 예정 |

### 7.1. 체크포인트

- [ ] Workaround를 적용한 뒤에도 서버가 여전히 취약했던 이유는 무엇인가요? Workaround로는 막지 못하는 경로는 어디인가요?
- [ ] 고장 난 패치가 Canary 한 대에서 멈추지 않고 세 대에 한꺼번에 배포되었다면 어떤 일이 생겼을까요?
- [ ] app3의 조치 누락은 무엇으로 찾아냈나요? 외부 스캐닝만으로 확인했다면 찾을 수 있었을까요?
- [ ] 조치가 확인되기 전에 Workaround를 해제하지 못하게 한 이유는 무엇인가요?

### 7.2. 과제 1: 티켓이 닫힌 뒤에도 다시 점검하기

티켓이 종결된 뒤, 누군가 장애 대응 중에 app2를 예전 이미지(1.0)로 몰래 되돌렸습니다. `patch_cycle.py`의 기록에는 이 일이 남지 않습니다. `patch_cycle.py`와 별개로 **매일 도는 독립 재점검** `verify_all()`을 완성해 이 서버를 찾아냅니다.

- 서버마다 `/version`과 `/admin/export`를 직접 불러, 영향 버전(`advisory.json`의 `affected`)이거나 공격이 통하면 "재발"로 표시합니다.
- 결과를 표로 보여 주고, 재발이 하나라도 있으면 `True`를 돌려줍니다.

In [17]:
import time
import urllib.error
import urllib.request

ADV = json.loads((WORK / "advisory.json").read_text())
PORT = {"app1": 8081, "app2": 8082, "app3": 8083}

# 누군가 app2를 몰래 1.0으로 되돌림 (patch_cycle.py 기록에는 남지 않음)
sh("docker rm -f app2 && docker run -d --name app2 --network patchlab -p 127.0.0.1:8082:8000 lab-app:1.0")
time.sleep(2)


def http_get(url):
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            return r.status, json.loads(r.read() or b"{}")
    except urllib.error.HTTPError as e:
        return e.code, {}


def verify_all():
    """서버마다 실제 버전과 공격 성공 여부를 다시 확인하고, 재발이 있으면 True를 돌려줍니다."""
    rows, regressed = [], False
    for s, port in PORT.items():
        # TODO: /version에서 버전을, /admin/export의 상태 코드로 취약 여부를 확인합니다.
        # TODO: 영향 버전이거나 취약하면 regressed = True, rows에 (서버, 버전, 취약 여부, 판정)을 넣습니다.
        pass
    table(["서버", "실행 중 버전", "공격 요청", "판정"], rows)
    return regressed


print("재발 발견:", verify_all())

| 서버 | 실행 중 버전 | 공격 요청 | 판정 |
|---|---|---|---|

재발 발견: False


<details>
<summary>💡 힌트 보기</summary>

```python
        _, v = http_get(f"http://127.0.0.1:{port}/version")
        code, _ = http_get(f"http://127.0.0.1:{port}/admin/export")
        bad = v.get("version") in ADV["affected"] or code == 200
        regressed |= bad
        rows.append((s, v.get("version"), "통함(200)" if code == 200 else f"막힘({code})", "⛔ 재발" if bad else "✅ 안전"))
```

티켓이 닫혔다고 위험이 끝난 것이 아닙니다. 구성은 언제든 되돌아갈 수 있으므로(설정 표류, 이미지 롤백, 복제한 옛 서버), 실습 1의 현황 수집과 이런 재점검을 **정기 작업**으로 돌리고, 재발이 나오면 티켓을 다시 엽니다. 이 랩을 다시 1.1로 맞추려면 `pc("patch", "1.1")`을 실행합니다.
</details>

### 7.3. 과제 2: Workaround 관리 규칙 (서술)

Workaround는 "임시"라고 붙이지만 실제로는 잊혀서 몇 년씩 남기도 합니다. 아래 질문에 답해 보세요.

1. Workaround를 등록할 때 반드시 함께 적어야 할 항목은 무엇일까요? (이 실습의 기록: 근거 공지, 규칙, 효과 확인 결과, 기한)
2. 기한이 지난 Workaround가 발견되면 누가 무엇을 해야 할까요?

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 8. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 환경 만들기와 현황 보기 | ✅ 완료 |
| 2 | 패치보다 먼저: Workaround 적용과 효과 확인 | ✅ 완료 |
| 3 | 고장 난 패치는 자동으로 되돌린다 | ✅ 완료 |
| 4 | "패치 완료" 보고를 믿지 않는다 | ✅ 완료 |
| 5 | 다시 배포하고, 확인되면 Workaround 해제 | ✅ 완료 |
| 6 | 지표로 읽기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | ✅ 완료 |
| 8 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [18]:
%%bash
docker rm -f app1 app2 app3 proxy >/dev/null 2>&1
docker network rm patchlab >/dev/null 2>&1
docker rmi lab-app:1.0 lab-app:1.1 lab-app:1.2 >/dev/null 2>&1
rm -rf ~/patch-lab && echo "컨테이너·네트워크·이미지·실습 폴더 삭제됨"

컨테이너·네트워크·이미지·실습 폴더 삭제됨


---
> 10장 실습에서는 이 장의 "현황 수집 → 대조 → 조치" 흐름을 AI 자산 전체로 넓혀, 정책을 코드로 검사하고(Policy as Code) 등록되지 않은 AI 사용(Shadow AI)을 찾습니다.